In [1]:
%matplotlib inline
import torch
from d2l import torch as d2l

torch.set_printoptions(2)

In [2]:
def multibox_prior(data, sizes, ratios):
    """生成以每个像素为中心具有不同形状的锚框"""
    in_height, in_width = data.shape[-2:]
    device, num_sizes, num_ratios = data.device, len(sizes), len(ratios)
    boxes_per_pixel = (num_sizes + num_ratios - 1)
    size_tensor = torch.tensor(sizes, device=device)
    ratio_tensor = torch.tensor(ratios, device=device)

    # 为了将锚点移动到像素的中心，需要设置偏移量
    offset_h, offset_w = 0.5, 0.5
    steps_h = 1.0 / in_height  # y 轴步长
    steps_w = 1.0 / in_width   # x 轴步长

    # ① 生成所有中心点坐标（归一化到 [0,1]）
    center_h = (torch.arange(in_height, device=device) + offset_h) * steps_h
    center_w = (torch.arange(in_width, device=device) + offset_w) * steps_w
    shift_y, shift_x = torch.meshgrid(center_h, center_w, indexing='ij')
    shift_y, shift_x = shift_y.reshape(-1), shift_x.reshape(-1)

    # ② 计算每个锚框的宽高
    w = torch.cat((size_tensor * torch.sqrt(ratio_tensor[0]),
                   sizes[0] * torch.sqrt(ratio_tensor[1:])))\
                   * in_height / in_width
    h = torch.cat((size_tensor / torch.sqrt(ratio_tensor[0]),
                   sizes[0] / torch.sqrt(ratio_tensor[1:])))
    # /2 得到半宽半高
    anchor_manipulations = torch.stack((-w, -h, w, h)).T.repeat(
        in_height * in_width, 1) / 2

    # ③ 中心点 + 偏移 = (xmin, ymin, xmax, ymax)
    out_grid = torch.stack([shift_x, shift_y, shift_x, shift_y],
                           dim=1).repeat_interleave(boxes_per_pixel, dim=0)
    output = out_grid + anchor_manipulations

    return output.unsqueeze(0)

In [5]:
img = d2l.plt.imread('catdog.jpg')
h, w = img.shape[:2]

print(h, w)
X = torch.rand(size=(1, 3, h, w))
Y = multibox_prior(X, sizes=[0.75, 0.5, 0.25], ratios=[1, 2, 0.5])

Y.shape

561 728


torch.Size([1, 2042040, 4])

交并比

In [6]:
def box_iou(boxes1, boxes2):
    """计算两组边界框的成对 IoU"""
    box_area = lambda boxes: ((boxes[:, 2] - boxes[:, 0]) *
                              (boxes[:, 3] - boxes[:, 1]))
    # areas1: (N,)  areas2: (M,)
    areas1 = box_area(boxes1)
    areas2 = box_area(boxes2)

    # 交集左上角和右下角，shape 都是 (N, M, 2)
    inter_upperlefts = torch.max(boxes1[:, None, :2], boxes2[:, :2])
    inter_lowerrights = torch.min(boxes1[:, None, 2:], boxes2[:, 2:])
    # 交集宽高，负值 clamp 为 0（不相交）
    # tensor.clamp(min=0)
    # 所有小于 0 的值 → 变成 0
    # 大于等于 0 的值 → 不变
    inters = (inter_lowerrights - inter_upperlefts).clamp(min=0)
    # inter_areas: (N, M)
    inter_areas = inters[:, :, 0] * inters[:, :, 1]

    # 并集 = box1面积 + box2面积 - 交集
    union_areas = areas1[:, None] + areas2 - inter_areas

    return inter_areas / union_areas

NMS函数

In [ ]:
def nms(boxes, scores, iou_threshold):
    """对预测边界框的置信度进行排序，执行非极大值抑制"""
    B = torch.argsort(scores, dim=-1, descending=True)  # 按分数从高到低排序，得到索引
    keep = []  # 保留下来的边界框索引
    while B.numel() > 0:  # 还有剩余框
        i = B[0]           # 当前最高分框的索引
        keep.append(i)
        if B.numel() == 1: break   # 只剩一个了，结束
        # 最高分框 vs 所有剩余框的 IoU
        iou = box_iou(boxes[i, :].reshape(-1, 4),
                      boxes[B[1:], :].reshape(-1, 4)).reshape(-1)
        # 找出 IoU <= 阈值的框（不重叠的保留）
        inds = torch.nonzero(iou <= iou_threshold).reshape(-1)
        B = B[inds + 1]   # +1 补偿 B[1:] 的偏移
    return torch.tensor(keep, device=boxes.device)